# The calibration baseline

A new user puts the band on anywhere: rotated, maybe running the other way round. Before the model can read it, the
band's 8 virtual channels must line up with the ones the model was trained on. The calibration baseline is what a short
recording of the user is compared against: for a few **actions** (rest, fist, pointing, two fingers, index flexion,
wrist flexion), the spatial pattern each one leaves on the virtual ring, learnt from every training session.

```
every processed session: virtual.npy (T, 8), labels.npy (T/5, 22)
   │  1. actions: held hand states in the labels / cues   CalibBaseline.frames
   │  2. signature: per action, log RMS vs rest per channel CalibBaseline.signature
   │  3. align: 8 rotations x 2 directions per session     CalibBaseline.align, anchor
   ▼
data/processed/calibration/baseline.json (templates) + sessions.parquet (where each band sat)

a new user: the same actions, cued → signature → match → VirtualArray(rotation=θ, mirror=m)
```

`scripts/build_dataset.py` builds it after the datasets (`--calibration-only` to rebuild only it); the settings are
in `src/femg/config/calib_baseline.yaml`, the checks in `scripts/tests/test_calib_baseline.py`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts" / "tests"))

from test_calib_baseline import aligned, session_folders, signatures, templates_without

from femg.calib_baseline import FOLDER, Baseline, CalibBaseline
from femg.electrodes import ElectrodeLayout
from femg.normalize import Normalizer, hand_flexion
from femg.virtual_array import VirtualArray

CALIB = CalibBaseline.load()
ARRAY = VirtualArray.load()
DATA, INTERIM = ROOT / "data" / "processed", ROOT / "data" / "interim"  # virtual EMG, labels / the real EMG
ANGLES = ARRAY.angles
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#e87ba4", "#eda100", "#8a5cd1"]
pd.DataFrame(CALIB.actions).T

## 1. The actions in one session

No dataset shares the same cue list (NinaPro 17 movements, DB8 9, GREAT 5 grasps, EMG-FK none), so the hand postures
come from the **labels**: frames where the fingers involved are held still within a range (`still_per_s`). Wrist flexion comes
from NinaPro's **cue** (Exercise B, movement 13): the glove's wrist labels are not reliable enough (their neutral sits
anywhere in 0-1, and NinaPro's wrist cues and labels disagree for extension and deviation, which are left out until
that is checked). NinaPro DB2 s01, exercise 1:

In [ ]:
folder = DATA / "ninapro_db2" / "s01" / "E1"
meta = json.loads((folder / "meta.json").read_text())
labels, stimulus = np.load(folder / "labels.npy"), np.load(folder / "stimulus.npy")
frames = CALIB.frames(labels, meta["label_fs"], (f"{meta['dataset']}/{meta['session']}", stimulus))
t = np.arange(len(labels)) / meta["label_fs"]
view = t < 600

fig, axes = plt.subplots(2, 1, figsize=(14, 4.5), sharex=True)
joints = {"finger flexion": hand_flexion(labels), "wrist_flex": labels[:, meta["joints"].index("wrist_flex")]}
for ax, (name, x) in zip(axes, joints.items(), strict=True):
    ax.plot(t[view], x[view], color="#28282a", lw=0.6)
    ax.set_ylabel(name, rotation=0, ha="right", va="center")
    ax.set_ylim(-0.1, 1.3)
for i, (action, mask) in enumerate(frames.items()):
    ax = axes[1 if "cues" in CALIB.actions[action] else 0]
    ax.fill_between(t[view], 0, 1, where=mask[view], color=COLORS[i % len(COLORS)], alpha=0.35, lw=0, label=action)
for ax in axes:
    ax.legend(fontsize=7, loc="upper right", ncol=3)
axes[-1].set_xlabel("time (s)")
plt.show()
print({a: f"{m.sum() / meta['label_fs']:.0f} s" for a, m in frames.items()})

## 2. A session's signature

For each action: the mean log RMS of each virtual channel over its frames, minus the same at rest, minus the ring mean.
What is left is **where** around the forearm the action shows up, not how strong (a gain cancels in the difference).

In [ ]:
signature = CALIB.session_signature(folder)
fig, ax = plt.subplots(figsize=(9, 4))
for i, (action, pattern) in enumerate(signature.items()):
    ax.plot(ANGLES, pattern, "o-", color=COLORS[i % len(COLORS)], label=action)
ax.axhline(0, color="#a8a7a2", lw=0.5)
ax.set_xticks(ANGLES, [f"{a:g}°" for a in ANGLES])
ax.set(xlabel="virtual electrode (0 palmar, 90 radial, 180 dorsal)", ylabel="log RMS vs rest, minus ring mean")
ax.legend(fontsize=7)
plt.show()

## 3. Aligning every session

No dataset documents where its band's channel 1 sat, nor which way the channels run, so the same action peaks at a
different column in each session (left). The alignment matches every session to the mean over its 16 possible
placements (8 rotations × 2 directions), recomputes the mean, and repeats until nothing moves; then it turns the whole
frame so that wrist flexion peaks palmar, where the flexors are. Right: the same signatures once aligned. With a
single anchor, which way round is radial is consistent across sessions but not fixed anatomically.

In [ ]:
sigs, info = signatures(CALIB, session_folders(DATA))
baseline = CALIB.fit(sigs, {k: {"dataset": v["dataset"], "subject": v["subject"]} for k, v in info.items()})
aligned_sigs = aligned(CALIB, sigs, baseline.sessions)

keys = list(sigs)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
for ax, source, title in ((axes[0], sigs, "fist, as recorded"), (axes[1], aligned_sigs, "fist, aligned")):
    rows = np.array([source[k].get("fist", np.full(ARRAY.n_angles, np.nan)) for k in keys])
    ax.imshow(rows, cmap="RdBu_r", vmin=-0.8, vmax=0.8, aspect="auto")
    ax.set_xticks(range(ARRAY.n_angles), [f"{a:g}°" for a in ANGLES])
    ax.set_title(title)
axes[0].set_yticks(range(len(keys)), keys, fontsize=7)
plt.show()
baseline.sessions.drop(columns=["subject"]).round(2)

The table is also a check of the data: the two exercises of one NinaPro subject (recorded without taking the band off)
usually get the same placement (DB7 does not: its second exercise has no wrist cue, only the fist to go by); GREAT's
blocks agree within a day and differ between days (band put on again). A low margin means the session's patterns fit
several placements about equally.

## 4. The baseline

One template per action, the mean of the aligned signatures, with their spread across sessions. This is what
`build_dataset.py` saves in `data/processed/calibration/`:

In [ ]:
saved = DATA / FOLDER
if (saved / "baseline.json").exists():
    baseline = Baseline.load(saved)
    print("loaded", saved)
fig, ax = plt.subplots(figsize=(10, 4.5))
for i, (action, template) in enumerate(baseline.templates.items()):
    spread = baseline.spread[action]
    ax.plot(ANGLES, template, "o-", color=COLORS[i % len(COLORS)], label=f"{action} ({baseline.counts[action]})")
    ax.fill_between(ANGLES, template - spread, template + spread, color=COLORS[i % len(COLORS)], alpha=0.1)
ax.axhline(0, color="#a8a7a2", lw=0.5)
ax.set_xticks(ANGLES, [f"{a:g}°" for a in ANGLES])
ax.set(xlabel="virtual electrode (0 palmar, 90 radial, 180 dorsal)", ylabel="log RMS vs rest, minus ring mean")
ax.legend(fontsize=7)
plt.show()

Fist and wrist flexion share part of their shape (which channels activate at all), with the differences on top: the
alignment uses the whole pattern (section 5), the anatomical anchor the wrist flexion peak.

## 5. Calibrating a new band

A simulated user: NinaPro DB2 subject 2, left out of the baseline. Their band is put on turned by 135° and running the
other way round (we rebuild their virtual channels from the real EMG with `VirtualArray(rotation=135, mirror=True)`).
(The real EMG comes from the interim session.) The calibration only sees the actions' signature, and scores all 16 placements against the templates of the other
subjects:

In [ ]:
user = DATA / "ninapro_db2" / "s02" / "E1"
others = {k: s for k, s in aligned_sigs.items() if not k.startswith("ninapro_db2/s02/")}
templates = templates_without(CALIB, others, None)

meta = json.loads((user / "meta.json").read_text())
emg = Normalizer.emg(np.load(INTERIM / meta["key"] / "emg.npy", mmap_mode="r"), meta["emg_stats"])
inputs = ARRAY.inputs(ElectrodeLayout.from_meta(meta))
labels = np.load(user / "labels.npy")

stimulus = np.load(user / "stimulus.npy")
as_worn = CALIB.signature(ARRAY(emg, inputs).reshape(len(emg), -1), labels, meta, stimulus)
put_on = (3, True)  # 135 deg, the other way round: unknown to the calibration
misplaced = CALIB.signature(
    ARRAY(emg, inputs, rotation=put_on[0] * 45, mirror=put_on[1]).reshape(len(emg), -1), labels, meta, stimulus
)
match = CALIB.match(misplaced, templates)
before = CALIB.match(as_worn, templates)

placements = CALIB.placements(ARRAY.n_angles)
shared = [a for a in misplaced if a in templates]
y = np.concatenate([templates[a] for a in shared])
x = np.stack([misplaced[a] for a in shared])
scores = [np.corrcoef(CALIB.transform(x, *p).ravel(), y)[0, 1] for p in placements]
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.bar(
    range(len(placements)),
    scores,
    color=["#eb6834" if p == (match.steps, match.mirror) else "#a8a7a2" for p in placements],
)
ax.set_xticks(range(len(placements)), [f"{k * 45}°{' m' if m else ''}" for k, m in placements], fontsize=7)
ax.set(ylabel="correlation with the templates", title="the 16 placements of the misplaced band (m: mirrored)")
plt.show()
print(
    f"match: rotation {match.rotation_deg:g}°, mirror {match.mirror}",
    f"score {match.score:.2f}, margin {match.margin:.2f}",
)
print(
    "the misplacement, then the match = the placement of the band as worn:",
    CALIB.compose(put_on, (match.steps, match.mirror), ARRAY.n_angles) == (before.steps, before.mirror),
)

The match undoes the misplacement: applied after it, it lands where the subject's band, as worn, belongs in the
baseline frame. For a real band, the match becomes the band's matrix, and every later sample goes through it:

In [ ]:
M = ARRAY.matrix(inputs, rotation=match.rotation_deg, mirror=match.mirror)
print(
    "M:", M.shape, "= VirtualArray.matrix(positions, gains, rotation, mirror), the gains from the calibration recording"
)

## 6. Summary

* The baseline is built from the processed sessions alone: actions from the labels, signatures from `virtual.npy`.
* Aligning the sessions recovers a session moved on purpose (94 %) and puts a session the same way against the others'
  templates (94 %, median score 0.76, margin 0.21); it also says where each dataset's band sat, a check of the data.
* A new user: about a minute of cued actions, their signature, `CalibBaseline.match`, and the band's matrix with that
  rotation and direction.
* With 1-2 subjects per dataset the templates are provisional: rebuild after the full download
  (`build_dataset.py --calibration-only`). More actions (wrist extension, deviation) need NinaPro's wrist cues and
  glove labels checked first; they would also fix the direction anatomically.